In [0]:
%run ./00_config


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")

bronze = spark.table(f"{BRONZE}.health_indicators_raw")
latest_file = bronze.agg(F.max_by("_source_file", "_ingested_at")).first()[0]
raw = bronze.filter(F.col("_source_file") == latest_file)

expected = {"ref_date", "geo", "dguid", "age_group", "sex", "indicators",
            "characteristics", "uom", "vector", "value", "status", "symbol"}
missing = expected - set(raw.columns)
assert not missing, f"Missing columns: {missing}. Found: {raw.columns}"

raw_rows = raw.count()
print("Latest file:", latest_file, "| rows:", raw_rows)

In [0]:
silver = (raw
    .select(
        F.col("ref_date").cast("int").alias("year"),
        F.trim("geo").alias("geo"),
        F.col("dguid"),
        F.trim("age_group").alias("age_group"),
        F.trim("sex").alias("sex"),
        F.trim("indicators").alias("indicator"),
        F.trim("characteristics").alias("characteristic"),
        F.trim("uom").alias("uom"),
        F.col("vector"),
        F.col("value").cast("double").alias("value"),
        F.col("status"),
        F.col("symbol"),
        F.col("_source_file"))
    .dropDuplicates(["vector", "year"])
    .withColumn("_loaded_at", F.current_timestamp()))

total = silver.count()
print(f"silver rows: {total:,}")

In [0]:
this_year = datetime.now(timezone.utc).year
results = []

def check(rule, failed_rows, severity):
    status = "PASS" if failed_rows == 0 else "FAIL"
    results.append((run_id, run_date, table_id, rule, severity,
                    int(failed_rows), int(total), status))

check("not_null_keys",
      silver.filter(F.col("year").isNull() | F.col("geo").isNull()
                    | F.col("indicator").isNull() | F.col("characteristic").isNull()
                    | F.col("vector").isNull()).count(),
      "critical")
check("year_in_range",
      silver.filter(~F.col("year").between(2000, this_year)).count(),
      "critical")
check("source_duplicates", raw_rows - total, "warning")
check("percent_between_0_and_100",
      silver.filter(F.lower("uom").contains("percent")
                    & ~F.col("value").between(0, 100)).count(),
      "warning")
check("value_missing", silver.filter(F.col("value").isNull()).count(), "info")

dq = (spark.createDataFrame(results,
        "run_id string, run_date string, table_id string, rule string, "
        "severity string, failed_rows long, total_rows long, status string")
      .withColumn("checked_at", F.current_timestamp()))
dq.write.mode("append").saveAsTable(f"{OPS}.dq_results")
display(dq)

failed = [r[3] for r in results if r[4] == "critical" and r[7] == "FAIL"]
if failed:
    raise Exception(f"Critical data-quality rules failed: {failed}. Silver was NOT updated.")

In [0]:
target = f"{SILVER}.health_indicators"

if spark.catalog.tableExists(target):
    (DeltaTable.forName(spark, target).alias("t")
        .merge(silver.alias("s"), "t.vector = s.vector AND t.year = s.year")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
else:
    silver.write.format("delta").saveAsTable(target)
    # A Delta constraint: the table itself refuses impossible years from now on
    spark.sql(f"ALTER TABLE {target} ADD CONSTRAINT valid_year CHECK (year BETWEEN 2000 AND 2100)")

print("Silver rows now:", spark.table(target).count())

In [0]:
(spark.createDataFrame(
    [(run_id, run_date, table_id, latest_file, int(raw_rows), int(total))],
    "run_id string, run_date string, table_id string, "
    "source_file string, raw_rows long, silver_rows long")
 .withColumn("logged_at", F.current_timestamp())
 .write.mode("append").saveAsTable(f"{OPS}.ingest_log"))